# Detection Results Viewer

Use this notebook to inspect residual baseline metrics and visualize alarm positions against labelled 2023/2024 operating states.

In [13]:
from pathlib import Path
import json
import re

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"
FLAGS_DIR = PROJECT_ROOT / "data" / "interim" / "kelmarsh" / "flags"
FIGURES_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "figures"
TURBINES = [f"Kelmarsh_{i}" for i in range(1, 7)]

def list_experiments():
    if not EXPERIMENTS_DIR.exists():
        return []
    return sorted(p.name for p in EXPERIMENTS_DIR.iterdir() if (p / "metadata.json").exists())

def list_detection_settings(run_id):
    run_dir = EXPERIMENTS_DIR / run_id
    files = sorted(run_dir.glob("residual_baseline_performance_*.json"))
    return [p.stem.replace("residual_baseline_performance_", "") for p in files]

def parse_setting(setting):
    match = re.fullmatch(r"q(\d+)_c(\d+)", setting)
    if not match:
        return {"quantile": None, "min_consecutive": None}
    return {"quantile": int(match.group(1)) / 1000, "min_consecutive": int(match.group(2))}

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

runs = list_experiments()
if not runs:
    raise FileNotFoundError(f"No experiment metadata found in {EXPERIMENTS_DIR}")

DEFAULT_RUN_ID = runs[-1]
DEFAULT_SETTINGS = list_detection_settings(DEFAULT_RUN_ID)
DEFAULT_SETTING = DEFAULT_SETTINGS[-1] if DEFAULT_SETTINGS else None

## Global Detection Metrics

These metrics are computed for the selected run and setting. For the current `all6` run, they summarize all six turbines, not the turbine selected later in the timeline.

In [14]:
def load_selected_result(run_id, setting):
    if setting is None:
        raise FileNotFoundError(f"No residual baseline detection result found for run: {run_id}")
    run_dir = EXPERIMENTS_DIR / run_id
    metadata = read_json(run_dir / "metadata.json")
    performance = read_json(run_dir / f"residual_baseline_performance_{setting}.json")
    thresholds = pd.read_csv(run_dir / f"residual_baseline_thresholds_{setting}.csv")
    return metadata, performance, thresholds

def show_key_metrics(run_id, setting):
    clear_output(wait=True)
    metadata, performance, thresholds = load_selected_result(run_id, setting)
    setting_info = parse_setting(setting)
    display(Markdown(f"### Selected Result: `{run_id}` / `{setting}`"))
    display(pd.DataFrame([{
        "dataset_id": metadata.get("dataset_id"),
        "turbines": ", ".join(metadata.get("turbines", [])),
        "setting": setting,
        "quantile": setting_info["quantile"],
        "min_consecutive": setting_info["min_consecutive"],
        "best_epoch": metadata.get("best_epoch"),
        "best_val_loss": metadata.get("best_val_loss"),
    }]))

    event_metrics = performance["event_level"]
    point_metrics = performance["alarm_point_level"]
    episode_metrics = performance["alarm_episode_level"]
    metric_table = pd.DataFrame([
        {
            "level": "event",
            "meaning": "fault events detected/missed under current horizon rule",
            "total": event_metrics["total_events"],
            "detected_or_true": event_metrics["detected_events"],
            "missed_or_false": event_metrics["missed_events"],
            "success_rate": event_metrics["detection_rate"],
            "error_rate": event_metrics["miss_rate"],
        },
        {
            "level": "alarm point",
            "meaning": "individual alarm timestamps inside/outside event horizons",
            "total": point_metrics["total_alarm_points"],
            "detected_or_true": point_metrics["true_alarm_points"],
            "missed_or_false": point_metrics["false_alarm_points"],
            "success_rate": point_metrics["true_alarm_points"] / point_metrics["total_alarm_points"] if point_metrics["total_alarm_points"] else 0,
            "error_rate": point_metrics["false_alarm_point_rate"],
        },
        {
            "level": "alarm episode",
            "meaning": "true horizon episodes vs operational false alarm episodes",
            "total": episode_metrics["total_alarm_episodes"],
            "detected_or_true": episode_metrics["true_alarm_episodes"],
            "missed_or_false": episode_metrics.get("operational_false_alarm_episodes", episode_metrics["false_alarm_episodes"]),
            "success_rate": episode_metrics["true_alarm_episodes"] / episode_metrics.get("operational_alarm_opportunities", episode_metrics["total_alarm_episodes"]) if episode_metrics.get("operational_alarm_opportunities", episode_metrics["total_alarm_episodes"]) else 0,
            "error_rate": episode_metrics.get("operational_false_alarm_rate", episode_metrics["false_alarm_episode_rate"]),
            "raw_false_alarm_episode_rate": episode_metrics["false_alarm_episode_rate"],
            "non_operational_alarm_episodes": episode_metrics.get("non_operational_alarm_episodes", 0),
        },
    ])
    display(Markdown("### Key Metrics"))
    display(metric_table.style.format({"success_rate": "{:.2%}", "error_rate": "{:.2%}", "raw_false_alarm_episode_rate": "{:.2%}"}, na_rep=""))

    threshold_view = thresholds.rename(columns={"threshold": "abs_residual_threshold"}).copy()
    threshold_view["lower_error_threshold"] = -threshold_view["abs_residual_threshold"]
    threshold_view["upper_error_threshold"] = threshold_view["abs_residual_threshold"]
    threshold_view = threshold_view[["target", "quantile", "lower_error_threshold", "upper_error_threshold", "abs_residual_threshold", "validation_samples"]]
    display(Markdown("### Residual Thresholds"))
    display(threshold_view.style.format({
        "quantile": "{:.3f}",
        "lower_error_threshold": "{:.4f}",
        "upper_error_threshold": "{:.4f}",
        "abs_residual_threshold": "{:.4f}",
    }))

if HAS_WIDGETS:
    run_dropdown = widgets.Dropdown(options=runs, value=DEFAULT_RUN_ID, description="Run")
    setting_dropdown = widgets.Dropdown(options=DEFAULT_SETTINGS, value=DEFAULT_SETTING, description="Setting")

    def update_settings(change):
        new_settings = list_detection_settings(change["new"])
        setting_dropdown.options = new_settings
        setting_dropdown.value = new_settings[-1] if new_settings else None

    run_dropdown.observe(update_settings, names="value")
    display(widgets.HBox([run_dropdown, setting_dropdown]))
    metrics_output = widgets.interactive_output(show_key_metrics, {"run_id": run_dropdown, "setting": setting_dropdown})
    display(metrics_output)
else:
    RUN_ID = DEFAULT_RUN_ID
    SETTING = DEFAULT_SETTING
    show_key_metrics(RUN_ID, SETTING)

Output()

## 2023/2024 Status and Detection Timeline

This plot reads already-built `data/interim/kelmarsh/flags` parquet files and alarm episode CSV files. It does not read the raw 2GB SCADA CSV files.

In [15]:
def period_bounds(period):
    if period == "2023":
        return pd.Timestamp("2023-01-01"), pd.Timestamp("2024-01-01")
    if period == "2024":
        return pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
    raise ValueError("Period must be 2023 or 2024")

def load_flag_timeline(turbine_id):
    path = FLAGS_DIR / f"{turbine_id.lower()}_with_flags.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing flag file: {path}")
    flag_cols = ["in_maintenance", "in_communication", "in_manual_event", "in_event"]
    cols = ["Date and time", "turbine_id"] + flag_cols
    df = pd.read_parquet(path, columns=cols)
    df["Date and time"] = pd.to_datetime(df["Date and time"], errors="coerce")
    return df.dropna(subset=["Date and time"]).sort_values("Date and time")

def flag_intervals(df, flag_col, start, end):
    part = df.loc[(df["Date and time"] >= start) & (df["Date and time"] < end), ["Date and time", flag_col]].copy()
    if part.empty:
        return []
    part[flag_col] = part[flag_col].fillna(False).astype(bool)
    groups = part[flag_col].ne(part[flag_col].shift()).cumsum()
    intervals = []
    for _, group in part.loc[part[flag_col]].groupby(groups):
        s = group["Date and time"].iloc[0]
        e = group["Date and time"].iloc[-1] + pd.Timedelta(minutes=10)
        intervals.append((s, min(e, end)))
    return intervals

def load_alarm_episodes(run_id, setting, turbine_id, start, end):
    path = EXPERIMENTS_DIR / run_id / f"residual_baseline_alarm_episodes_{setting}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing alarm episode file: {path}")
    episodes = pd.read_csv(path)
    episodes["start_time"] = pd.to_datetime(episodes["start_time"], errors="coerce")
    episodes["end_time"] = pd.to_datetime(episodes["end_time"], errors="coerce")
    return episodes[
        (episodes["turbine_id"] == turbine_id)
        & episodes["start_time"].notna()
        & episodes["end_time"].notna()
        & (episodes["end_time"] >= start)
        & (episodes["start_time"] < end)
    ].copy().sort_values(["target", "start_time"])

def draw_interval_timeline(run_id, setting, turbine_id, period):
    clear_output(wait=True)
    start, end = period_bounds(period)
    flags = load_flag_timeline(turbine_id)
    episodes = load_alarm_episodes(run_id, setting, turbine_id, start, end)

    status_lanes = [
        ("Maintenance", "in_maintenance", "#ffd500"),
        ("Communication", "in_communication", "#00a6fb"),
        ("Manual event", "in_manual_event", "#d00000"),
        ("Status event/downtime", "in_event", "#ff7b00"),
    ]
    targets = list(episodes["target"].dropna().unique())
    lanes = [name for name, _, _ in status_lanes] + [f"Alarm: {target.split('(')[0].strip()}" for target in targets]

    fig_height = max(4.5, 0.48 * len(lanes) + 2.0)
    fig, ax = plt.subplots(figsize=(15, fig_height), facecolor="white")
    ax.set_facecolor("white")
    y_positions = {lane: len(lanes) - 1 - i for i, lane in enumerate(lanes)}
    bar_height = 0.72

    for lane_name, flag_col, color in status_lanes:
        y = y_positions[lane_name]
        for s, e in flag_intervals(flags, flag_col, start, end):
            ax.broken_barh(
                [(mdates.date2num(s), max(mdates.date2num(e) - mdates.date2num(s), 10 / 1440))],
                (y - bar_height / 2, bar_height),
                facecolors=color,
                edgecolors="black",
                linewidth=0.25,
                alpha=0.9,
            )

    for target in targets:
        lane_name = f"Alarm: {target.split('(')[0].strip()}"
        y = y_positions[lane_name]
        target_eps = episodes.loc[episodes["target"] == target]
        for _, row in target_eps.iterrows():
            s = max(row["start_time"], start)
            e = min(row["end_time"] + pd.Timedelta(minutes=10), end)
            color = "#008000" if bool(row.get("in_fault_horizon", False)) else "#111111"
            ax.broken_barh(
                [(mdates.date2num(s), max(mdates.date2num(e) - mdates.date2num(s), 10 / 1440))],
                (y - bar_height / 2, bar_height),
                facecolors=color,
                edgecolors="black",
                linewidth=0.2,
                alpha=0.88,
            )

    ax.set_xlim(mdates.date2num(start), mdates.date2num(end))
    ax.set_yticks([y_positions[lane] for lane in lanes])
    ax.set_yticklabels(lanes)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.grid(axis="x", color="#d0d0d0", alpha=0.8)
    ax.set_title(f"{turbine_id}: labelled intervals and detection alarms ({period}, {setting})")
    ax.set_xlabel("Time")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    legend_handles = [
        plt.Line2D([0], [0], color="#d00000", lw=6, label="Manual event"),
        plt.Line2D([0], [0], color="#ff7b00", lw=6, label="Status event/downtime"),
        plt.Line2D([0], [0], color="#ffd500", lw=6, label="Maintenance"),
        plt.Line2D([0], [0], color="#00a6fb", lw=6, label="Communication"),
        plt.Line2D([0], [0], color="#008000", lw=6, label="Alarm overlaps horizon"),
        plt.Line2D([0], [0], color="#111111", lw=6, label="Alarm outside horizon"),
    ]
    ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)
    plt.tight_layout()

    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    safe_run_id = str(run_id).replace("/", "_").replace("\\", "_")
    timeline_png_path = FIGURES_DIR / f"detection_timeline_{safe_run_id}_{setting}_{turbine_id}_{period}.png"
    timeline_pdf_path = timeline_png_path.with_suffix(".pdf")
    fig.savefig(timeline_png_path, dpi=300, bbox_inches="tight")
    fig.savefig(timeline_pdf_path, bbox_inches="tight")
    print(f"Saved timeline figure to: {timeline_png_path}")
    print(f"Saved PDF figure to: {timeline_pdf_path}")

    plt.show()

    summary = pd.DataFrame([{
        "turbine": turbine_id,
        "period": period,
        "alarm episodes": len(episodes),
        "alarm episodes overlapping horizon": int(episodes.get("in_fault_horizon", pd.Series(dtype=bool)).fillna(False).sum()),
        "alarm episodes outside horizon": int((~episodes.get("in_fault_horizon", pd.Series(dtype=bool)).fillna(False)).sum()) if len(episodes) else 0,
    }])
    display(summary)

if HAS_WIDGETS:
    timeline_turbine_dropdown = widgets.Dropdown(options=TURBINES, value="Kelmarsh_1", description="Turbine")
    timeline_year_dropdown = widgets.Dropdown(options=["2023", "2024"], value="2023", description="Period")
    display(widgets.HBox([timeline_turbine_dropdown, timeline_year_dropdown]))
    timeline_output = widgets.interactive_output(
        draw_interval_timeline,
        {
            "run_id": run_dropdown,
            "setting": setting_dropdown,
            "turbine_id": timeline_turbine_dropdown,
            "period": timeline_year_dropdown,
        },
    )
    display(timeline_output)
else:
    TURBINE = "Kelmarsh_1"
    PERIOD = "2023"
    draw_interval_timeline(RUN_ID, SETTING, TURBINE, PERIOD)
    




Output()